# Neuronal Coactivity by Vocal Category — Analyses

This notebook quantifies how coordinated a PAG (or other region) neural population is during one class of ultrasonic vocalizations (USVs) versus another, using the pairwise spike-count correlation, population-vector cosine similarity, and population-vector Pearson correlation computed by `usv_playpen.analyses.neuronal_coactivity_engine`.

The workflow is: pooled trial-count bootstrap to a matched N, a chained circular-shuffle null per group, and a direct group-A-vs-group-B label-permutation test — reported as summary tables, per-metric null-distribution plots, a per-session breakdown, and a cross-animal slope plot.

The cells are organised as *(1) imports*, *(2) parameters — every tweakable knob*, *(3) setup & data load*, then *(4) per-section compute + plot*. Each compute/plot cell consumes the **Imports**, **Parameters**, and **Setup & load** cells; re-run those first.

In [ ]:
### Imports

import csv
import pathlib
from collections import defaultdict

import h5py
import matplotlib.pyplot as plt
import numpy as np
import polars as pls

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.plot_style import apply_plot_style
import usv_playpen.analyses.neuronal_coactivity_engine as engine

apply_plot_style()

## Parameters

Every knob a user might tweak for a run lives in the single cell below — trial segmentation, unit filters, data paths, the animal→sessions map, the coactivity hyperparameters, and the per-group plotting colours. Edit here; nothing downstream redefines these. Paths are written `/mnt/falkner/...` and wrapped in `configure_path()` so they resolve on macOS (`/Volumes/falkner`) too.

In [ ]:
### Parameters — every user-tweakable knob lives here

# Segmentation configuration
CATEGORY_COLUMN = "qlvm_supercategory"
GROUP_A_IDS   = [1]
GROUP_A_LABEL = "complex"
GROUP_B_IDS   = [7]
GROUP_B_LABEL = "simple"

# Unit-filter configuration (3 criteria: cluster_group + somatic + brain area)
CATALOG_PATH       = pathlib.Path(configure_path("/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"))
UNIT_BRAIN_AREAS   = {"PAG"}
UNIT_REQUIRE_SOMATIC = True
UNIT_CLUSTER_GROUP = "good"

# Animal -> sessions mapping
# Sessions can span multiple recording days. Kilosort is run per day,
# so cluster IDs aren't stable across days; the loader picks the
# single-day block with the MOST units passing the catalog filters,
# breaking ties by session count. Days where the probe wasn't in the
# requested brain area (post-hoc histology) are automatically skipped.
DATA_ROOT = pathlib.Path(configure_path("/mnt/falkner/Bartul/Data"))

ANIMALS_TO_SESSIONS: dict[str, list[str]] = {
    "158112_0": [
        "20241107_135544", "20241107_143336", "20241107_154636",
        "20241107_172830",
    ],
    "158114_2": [
        "20241115_165038", "20241115_172055", "20241115_185527",
        "20241115_192700",
    ],
    "164335_0": [
        "20250909_151929", "20250909_154745",
        "20250911_154739", "20250911_162235", "20250911_165912",
        "20250911_182910", "20250911_190415", "20250911_194127",
        "20250912_125331", "20250912_150922", "20250912_155546",
        "20250912_174643", "20250912_181920", "20250912_185338",
        "20250913_143737", "20250913_150744", "20250913_153700",
        "20250913_173732", "20250913_180808", "20250913_183548",
    ],
    "181316_0": [
        "20250919_152924", "20250919_155842", "20250919_163351",
        "20250923_162249", "20250923_171803", "20250923_174642",
        "20250923_184326", "20250923_200538", "20250923_203320",
    ],
    "178621_2": [
        "20250927_142335", "20250927_145144", "20250927_151825",
        "20250927_172337", "20250927_175128", "20250927_181936",
        "20250928_172408", "20250928_175135", "20250928_182348",
        "20250928_192858", "20250928_195647", "20250928_202420",
    ],
    "181321_1": [
        "20251003_140135", "20251003_143416", "20251003_150452",
        "20251003_161312", "20251003_164534", "20251003_171558",
        "20251004_154544", "20251004_162927", "20251004_170028",
        "20251004_180923", "20251004_183841", "20251004_190735",
    ],
    "181322_2": [
        "20251011_140347", "20251011_145914", "20251011_153450",
        "20251011_174924", "20251011_182556", "20251011_190000",
        "20251012_150953", "20251012_154216", "20251012_161241",
        "20251012_171752", "20251012_174634", "20251012_181640",
    ],
}

CHOSEN_ANIMAL = "178621_2"

# Coactivity hyperparameters
USV_BOOTSTRAP_NUM = 300
N_BOOT_ITERATIONS = 1000
N_SHUFFLES = 1000
N_PERMUTATIONS = 1000
WINDOW_S = 0.030
PER_SESSION_N_SHUFFLES = 500   # smaller than the chained null since we run per session

# Group plotting colours (hex). Group A uses crimson, group B uses dodgerblue
# by default — change here to retune. Labels come from the segmentation config above.
GROUP_A_COLOR = "#DC143C"   # crimson
GROUP_B_COLOR = "#1E90FF"   # dodgerblue
NULL_COLOR    = "#808080"   # grey
THRESHOLD_COLOR = "#000000"

## Setup & load data

Helper functions that read the unit catalog, apply the three-criteria unit filter, and assemble per-animal session data; then load the chosen animal. You should not need to edit this cell — change inputs in **Parameters** above.

In [ ]:
### Setup & load data

def _load_catalog() -> dict[tuple[str, str, str], dict]:
    """
    Read the unit catalog into a (mouse_id, rec_date, unit_id) ->
    row dict so per-session filtering is an O(1) lookup. The catalog
    is keyed by recording date, so all sessions on that date for that
    mouse share the same row set.

    Returns
    -------
    catalog (dict)
        Lookup keyed by `(mouse_id, rec_date, unit_id)` with the raw
        CSV row dict as value. All three components are strings.
    """

    out: dict[tuple[str, str, str], dict] = {}
    with open(CATALOG_PATH) as fh:
        for row in csv.DictReader(fh):
            out[(row["mouse_id"], row["rec_date"], row["unit_id"])] = row
    return out


def _filter_units_by_catalog(
    mouse_id: str,
    rec_date: str,
    file_stems: set[str],
    catalog: dict[tuple[str, str, str], dict],
) -> set[str]:
    """
    Keep only `file_stems` whose catalog row passes the three
    configured filters: `cluster_group`, `somatic` (when required),
    and `brain_area` (when `UNIT_BRAIN_AREAS` is non-empty). Stems
    without a catalog row are silently dropped.

    Parameters
    ----------
    mouse_id (str)
        Focal mouse ID.
    rec_date (str)
        Recording date in `YYYYMMDD` form.
    file_stems (set[str])
        Cluster file stems found in the session directory.
    catalog (dict)
        Output of `_load_catalog`.

    Returns
    -------
    kept (set[str])
        Subset of `file_stems` passing every filter.
    """

    kept: set[str] = set()
    for stem in file_stems:
        row = catalog.get((mouse_id, rec_date, stem))
        if row is None:
            continue
        if row["cluster_group"] != UNIT_CLUSTER_GROUP:
            continue
        if UNIT_REQUIRE_SOMATIC and str(row["somatic"]).strip().lower() != "true":
            continue
        if UNIT_BRAIN_AREAS and row["brain_area"] not in UNIT_BRAIN_AREAS:
            continue
        kept.add(stem)
    return kept


def load_animal_sessions(animal_id: str, session_names: list[str]) -> list[dict]:
    """
    Build the `sessions_data` list for one focal mouse, restricted to
    the single recording day with the LARGEST filtered-unit pool so
    the neural population is fixed across the sessions actually
    analysed AND the probe is in the requested brain area. Per-session
    sessions of that day get the three-criteria catalog filter; the
    intersection across that day's sessions becomes the common unit
    set.

    Day-scoring: each candidate date is scored by the size of the
    catalog-filtered unit set on a representative (first) session of
    that date. The day with the largest score wins. Ties are broken
    by session count.

    Each entry carries: per-session USV onsets split into the two
    groups, spike-time arrays for the filtered+common unit set, the
    recording frame rate, and the total session duration. The tracks
    array itself is not materialised — only its leading dimension is
    read so `total_duration = n_frames / fs` can be computed without
    paying the H5 read cost.

    Parameters
    ----------
    animal_id (str)
        Focal-mouse ID (matches `track_names[0]` in the tracking H5).
    session_names (list[str])
        Session directory names under `DATA_ROOT` for this animal —
        can span multiple recording days; the loader picks the
        single-day block with the largest filtered unit count.

    Returns
    -------
    sessions_data (list[dict])
        One dict per session of the chosen day.
    """

    catalog = _load_catalog()

    # Group by date.
    by_date: dict[str, list[str]] = defaultdict(list)
    for s in session_names:
        by_date[s.split("_")[0]].append(s)
    if not by_date:
        return []

    # Score each date by the size of its catalog-filtered unit set
    # (using the first session of that date as representative). Days
    # where the probe wasn't in the requested brain area score 0 and
    # are automatically skipped in favour of days where it was. Ties
    # broken by session count.
    def _score(date: str) -> tuple[int, int]:
        first_sess = sorted(by_date[date])[0]
        on_disk = {f.stem for f in (DATA_ROOT / first_sess).glob("**/*_good.npy")}
        filtered = _filter_units_by_catalog(animal_id, date, on_disk, catalog)
        return (len(filtered), len(by_date[date]))

    scored = {d: _score(d) for d in by_date}
    chosen_date = max(scored, key=lambda d: scored[d])
    chosen_session_names = by_date[chosen_date]
    print(
        f"  {animal_id}: picked day {chosen_date} "
        f"({scored[chosen_date][0]} filtered units, "
        f"{scored[chosen_date][1]} sessions); "
        f"all days = " + ", ".join(
            f"{d}({s[0]}u/{s[1]}s)" for d, s in sorted(scored.items())
        )
    )
    session_dirs = [DATA_ROOT / s for s in chosen_session_names]

    # Per-session unit set: on-disk `_good.npy` files filtered by
    # catalog rules. Intersect across the chosen day's sessions.
    per_session_sets: list[set[str]] = []
    per_session_rec_dates: list[str] = []
    for sdir in session_dirs:
        rec_date = sdir.name.split("_")[0]
        per_session_rec_dates.append(rec_date)
        on_disk = {f.stem for f in sdir.glob("**/*_good.npy")}
        per_session_sets.append(
            _filter_units_by_catalog(animal_id, rec_date, on_disk, catalog)
        )
    common_unit_ids = (
        set.intersection(*per_session_sets) if per_session_sets else set()
    )
    print(f"  {animal_id}: common filtered units = {len(common_unit_ids)}")

    sessions_data = []
    for directory, rec_date in zip(session_dirs, per_session_rec_dates):
        tracking_file = next(directory.glob('**/*_translated_rotated_metric.h5'))
        with h5py.File(name=tracking_file, mode='r') as track_file:
            mouse_track_names = [t.decode('utf-8') for t in list(track_file['track_names'])]
            recording_frame_rate = float(track_file['recording_frame_rate'][()])
            n_frames = int(track_file['tracks'].shape[0])

        usv_summary_file = next(directory.glob('**/*_usv_summary.csv'))
        usv_summary_data = pls.read_csv(usv_summary_file)
        focal_usvs = usv_summary_data.filter(pls.col('emitter') == mouse_track_names[0])
        group_a_df = focal_usvs.filter(pls.col(CATEGORY_COLUMN).is_in(GROUP_A_IDS))
        group_b_df = focal_usvs.filter(pls.col(CATEGORY_COLUMN).is_in(GROUP_B_IDS))

        session_neural_data = {
            unit_id: np.load(next(directory.glob(f'**/{unit_id}.npy')))[0, :]
            for unit_id in common_unit_ids
        }

        sessions_data.append({
            'session_id':     directory.name,
            'fs':             recording_frame_rate,
            'group_a_df':     group_a_df,
            'group_b_df':     group_b_df,
            'neural_data':    session_neural_data,
            'total_duration': n_frames / recording_frame_rate,
        })
    return sessions_data


# Load the chosen animal's data
print(f"Trial split:  `{CATEGORY_COLUMN}`")
print(f"  group A ({GROUP_A_LABEL}) = IDs {GROUP_A_IDS}")
print(f"  group B ({GROUP_B_LABEL}) = IDs {GROUP_B_IDS}")
print(f"Unit filter:  cluster_group='{UNIT_CLUSTER_GROUP}'  "
      f"somatic={UNIT_REQUIRE_SOMATIC}  brain_area in {sorted(UNIT_BRAIN_AREAS) or 'ANY'}")
print(f"Chosen animal: {CHOSEN_ANIMAL}")

sessions_data = load_animal_sessions(CHOSEN_ANIMAL, ANIMALS_TO_SESSIONS[CHOSEN_ANIMAL])
n_common = (
    len(next(iter(sessions_data))['neural_data']) if sessions_data else 0
)
print(
    f"Loaded {len(sessions_data)} sessions for {CHOSEN_ANIMAL}; "
    f"common filtered units = {n_common}"
)

## Compute coactivity metrics: group A vs group B

In [ ]:
### Compute coactivity metrics: group A vs group B

# 1. Per-session observed count matrices, then trial-axis concat
group_a_mats, group_b_mats = [], []
for sess in sessions_data:
    a_mat = engine.extract_snippet_matrix(
        sess['group_a_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    b_mat = engine.extract_snippet_matrix(
        sess['group_b_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    group_a_mats.append(a_mat)
    group_b_mats.append(b_mat)

all_group_a_counts = np.hstack(group_a_mats)
all_group_b_counts = np.hstack(group_b_mats)
print(
    f"Aggregated: {all_group_a_counts.shape[1]} {GROUP_A_LABEL} | "
    f"{all_group_b_counts.shape[1]} {GROUP_B_LABEL} trials"
)

# 2. Pooled bootstrap to matched N
print(f"Bootstrapping pooled data to N={USV_BOOTSTRAP_NUM} ...")
all_group_a_boot = engine.bootstrap_coactivity_distribution(
    all_group_a_counts, USV_BOOTSTRAP_NUM, N_BOOT_ITERATIONS,
)
all_group_b_boot = engine.bootstrap_coactivity_distribution(
    all_group_b_counts, USV_BOOTSTRAP_NUM, N_BOOT_ITERATIONS,
)

# 3. Chained circular-shuffle null per group
print(f"Generating matched-N chained null distributions ...")
session_group_a_onsets = engine.sample_onsets_across_sessions(
    sessions_data, 'group_a_df', USV_BOOTSTRAP_NUM,
)
session_group_b_onsets = engine.sample_onsets_across_sessions(
    sessions_data, 'group_b_df', USV_BOOTSTRAP_NUM,
)

group_a_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_group_a_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=N_SHUFFLES,
)
group_b_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_group_b_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=N_SHUFFLES,
)

# 4. Direct A-vs-B permutation test
# Tests `metric(A) - metric(B)` against a null built by shuffling
# group labels across the pooled trial set. Unlike the bootstrap-vs-
# shuffle test above (which only compares each group to its own
# random-timing null), this directly answers "do the two groups
# differ in coactivity?".
print(f"Running label permutation test (n_perm={N_PERMUTATIONS}) ...")
perm_results = engine.perform_label_permutation_test(
    counts_a=all_group_a_counts,
    counts_b=all_group_b_counts,
    n_permutations=N_PERMUTATIONS,
)

# 5. Per-session breakdown — does the effect hold session-by-session?
print("\nPer-session observed metrics (no bootstrap / no shuffle):")
print(
    f"  {'session':<22} {'n_a':>4} {'n_b':>4}"
    f"   {'r_sc Δ':>10}   {'sim Δ':>10}   {'pop Δ':>10}"
)
per_session_deltas = {m: [] for m in ('r_sc', 'similarity', 'pop_corr')}
for sess in sessions_data:
    a_mat = engine.extract_snippet_matrix(
        sess['group_a_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    b_mat = engine.extract_snippet_matrix(
        sess['group_b_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S,
    )
    if a_mat.shape[1] < 2 or b_mat.shape[1] < 2:
        print(
            f"  {sess['session_id']:<22} {a_mat.shape[1]:>4} {b_mat.shape[1]:>4}"
            "   (skipped, too few trials)"
        )
        continue
    m_a = engine.compute_coactivity_metrics(a_mat)
    m_b = engine.compute_coactivity_metrics(b_mat)
    deltas = {k: m_a[k] - m_b[k] for k in m_a}
    for k in per_session_deltas:
        per_session_deltas[k].append(deltas[k])
    print(
        f"  {sess['session_id']:<22} {a_mat.shape[1]:>4} {b_mat.shape[1]:>4}"
        f"   {deltas['r_sc']:>+10.4f}"
        f"   {deltas['similarity']:>+10.4f}"
        f"   {deltas['pop_corr']:>+10.4f}"
    )

print(f"\nSessions where {GROUP_A_LABEL} > {GROUP_B_LABEL}, per metric:")
for k, vals in per_session_deltas.items():
    n_pos = sum(1 for v in vals if v > 0)
    print(f"  {k:<12}: {n_pos}/{len(vals)} sessions")

# 6. Summary tables

def calculate_stats(boot_data, null_data):
    """Right-tailed empirical p of boot vs shuffle null + Z-score."""
    boot_mean = float(np.mean(boot_data))
    null_mean = float(np.mean(null_data))
    null_std = float(np.std(null_data))
    p_val = float(np.mean(null_data >= boot_mean))
    z = (boot_mean - null_mean) / null_std if null_std > 0 else 0.0
    return boot_mean, null_mean, p_val, z

print("\n" + "=" * 75)
print(f"{GROUP_A_LABEL.upper()} vs CHAINED NULL  (N={USV_BOOTSTRAP_NUM})")
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    obs, null, p, z = calculate_stats(all_group_a_boot[m], group_a_chained_null[m])
    print(f"  {m:<12}: boot={obs:+.4f} | null={null:+.4f} | p={p:.4f}  (Z={z:+.2f})")

print(f"\n{GROUP_B_LABEL.upper()} vs CHAINED NULL  (N={USV_BOOTSTRAP_NUM})")
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    obs, null, p, z = calculate_stats(all_group_b_boot[m], group_b_chained_null[m])
    print(f"  {m:<12}: boot={obs:+.4f} | null={null:+.4f} | p={p:.4f}  (Z={z:+.2f})")

print("\n" + "=" * 75)
print(
    f"DIRECT PERMUTATION: {GROUP_A_LABEL.upper()} vs {GROUP_B_LABEL.upper()}"
    f"  (n_perm={N_PERMUTATIONS})"
)
print("=" * 75)
for m in ('r_sc', 'similarity', 'pop_corr'):
    r = perm_results[m]
    print(
        f"  Δ {m:<12}: obs={r['observed_delta']:+.4f} | "
        f"null mean={r['null_mean']:+.4f} | "
        f"p({GROUP_A_LABEL}>{GROUP_B_LABEL})={r['p_a_gt_b']:.4f} | "
        f"p_two_tail={r['p_two_tailed']:.4f}  (Z={r['z_score']:+.2f})"
    )

## Plot per-metric chained-null distributions with observed bootstrap means

In [ ]:
### Plot per-metric chained-null distributions with observed bootstrap means

# 1. 99th-percentile thresholds from each chained null
a_rsc_99 = np.percentile(group_a_chained_null['r_sc'], 99)
a_sim_99 = np.percentile(group_a_chained_null['similarity'], 99)
a_pop_99 = np.percentile(group_a_chained_null['pop_corr'], 99)

b_rsc_99 = np.percentile(group_b_chained_null['r_sc'], 99)
b_sim_99 = np.percentile(group_b_chained_null['similarity'], 99)
b_pop_99 = np.percentile(group_b_chained_null['pop_corr'], 99)

# 2. Observed pooled-bootstrap means
a_val_rsc = np.mean(all_group_a_boot['r_sc'])
a_val_sim = np.mean(all_group_a_boot['similarity'])
a_val_pop = np.mean(all_group_a_boot['pop_corr'])

b_val_rsc = np.mean(all_group_b_boot['r_sc'])
b_val_sim = np.mean(all_group_b_boot['similarity'])
b_val_pop = np.mean(all_group_b_boot['pop_corr'])

# 3. 3 rows (metric) x 2 columns (group)
fig, axes = plt.subplots(3, 2, figsize=(14, 15), sharey='row')

panels = [
    ("Pairwise Correlation ($r_{sc}$)",  'r_sc',       a_rsc_99, a_val_rsc, b_rsc_99, b_val_rsc),
    ("Cosine Similarity",                'similarity', a_sim_99, a_val_sim, b_sim_99, b_val_sim),
    ("Pop Vector Corr (Pearson)",        'pop_corr',   a_pop_99, a_val_pop, b_pop_99, b_val_pop),
]

for row, (panel_title, key, a99, a_val, b99, b_val) in enumerate(panels):
    # Group A
    ax = axes[row, 0]
    ax.hist(group_a_chained_null[key], bins=40, color=NULL_COLOR, alpha=0.5, label='Chained Null')
    ax.axvline(a99,   color=THRESHOLD_COLOR, linestyle='--', label='99%')
    ax.axvline(a_val, color=GROUP_A_COLOR,   linewidth=3, label=f'Pooled Boot ({a_val:.4f})')
    ax.set_title(f'{GROUP_A_LABEL.upper()}: {panel_title}', fontsize=14)
    ax.legend(fontsize=8)
    if row == 2:
        ax.set_xlabel('Mean correlation / similarity', fontsize=12)

    # Group B
    ax = axes[row, 1]
    ax.hist(group_b_chained_null[key], bins=40, color=NULL_COLOR, alpha=0.5, label='Chained Null')
    ax.axvline(b99,   color=THRESHOLD_COLOR, linestyle='--', label='99%')
    ax.axvline(b_val, color=GROUP_B_COLOR,   linewidth=3, label=f'Pooled Boot ({b_val:.4f})')
    ax.set_title(f'{GROUP_B_LABEL.upper()}: {panel_title}', fontsize=14)
    ax.legend(fontsize=8)
    if row == 2:
        ax.set_xlabel('Mean correlation / similarity', fontsize=12)

# Cleanup spines
for ax_row in axes:
    for ax in ax_row:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

fig.suptitle(
    f"Coactivity by `{CATEGORY_COLUMN}`  ·  "
    f"{GROUP_A_LABEL} (IDs={GROUP_A_IDS}) vs {GROUP_B_LABEL} (IDs={GROUP_B_IDS})",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Per-session pop_corr — how the metric behaves session-by-session

In [ ]:
### Per-session pop_corr — how the metric behaves session-by-session

# For the chosen animal, plot one panel per session showing:
#   * histogram = within-session circular-shuffle null distribution of
#     pop_corr (combined across the two groups' onset templates so the
#     null reflects neural-data shifts, not category-specific timing).
#   * vertical line in GROUP_A_COLOR = observed pop_corr for group A
#     in that session.
#   * vertical line in GROUP_B_COLOR = observed pop_corr for group B
#     in that session.
#   * dashed line at the 99th percentile of the null.
#
# Sessions with too few trials in either group (<2) are skipped.



valid_session_rows = []
for sess in sessions_data:
    a_starts = sess['group_a_df']['start'].to_numpy()
    b_starts = sess['group_b_df']['start'].to_numpy()
    if len(a_starts) < 2 or len(b_starts) < 2:
        continue
    a_mat = engine.extract_snippet_matrix(a_starts, sess['neural_data'], WINDOW_S)
    b_mat = engine.extract_snippet_matrix(b_starts, sess['neural_data'], WINDOW_S)
    pop_a = engine.compute_coactivity_metrics(a_mat)['pop_corr']
    pop_b = engine.compute_coactivity_metrics(b_mat)['pop_corr']
    # single-session circular shuffle null using BOTH groups' onsets so
    # the null sample size matches the joint trial count.
    joint_onsets = np.concatenate([a_starts, b_starts])
    null_distribution = engine.perform_circular_shuffle(
        onsets=joint_onsets,
        neural_data=sess['neural_data'],
        total_duration=sess['total_duration'],
        window_s=WINDOW_S,
        n_shuffles=PER_SESSION_N_SHUFFLES,
    )
    valid_session_rows.append({
        'session_id': sess['session_id'],
        'n_a': len(a_starts), 'n_b': len(b_starts),
        'pop_a': pop_a, 'pop_b': pop_b,
        'null': null_distribution['pop_corr'],
    })

if not valid_session_rows:
    print("No sessions with sufficient trials in BOTH groups; figure skipped.")
else:
    n_panels = len(valid_session_rows)
    n_cols = min(4, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(4.0 * n_cols, 3.0 * n_rows),
        sharex=False, sharey=False, squeeze=False,
    )
    for idx, row in enumerate(valid_session_rows):
        ax = axes[idx // n_cols, idx % n_cols]
        ax.hist(row['null'], bins=30, color=NULL_COLOR, alpha=0.55)
        thr99 = float(np.percentile(row['null'], 99))
        ax.axvline(thr99, color=THRESHOLD_COLOR, linestyle='--', linewidth=0.8)
        ax.axvline(
            row['pop_a'], color=GROUP_A_COLOR, linewidth=2.2,
            label=f"{GROUP_A_LABEL} (n={row['n_a']})  {row['pop_a']:.3f}",
        )
        ax.axvline(
            row['pop_b'], color=GROUP_B_COLOR, linewidth=2.2,
            label=f"{GROUP_B_LABEL} (n={row['n_b']})  {row['pop_b']:.3f}",
        )
        ax.set_title(row['session_id'], fontsize=9)
        ax.set_xlabel('pop_corr', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=7, loc='upper right')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    # blank any unused panels
    for idx in range(n_panels, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].axis('off')

    fig.suptitle(
        f"Per-session pop_corr — {CHOSEN_ANIMAL}  ·  "
        f"{GROUP_A_LABEL} vs {GROUP_B_LABEL} ({CATEGORY_COLUMN})",
        fontsize=11,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

## Cross-animal pop_corr summary

In [ ]:
### Cross-animal pop_corr summary

# Loops over every focal mouse in `ANIMALS_TO_SESSIONS`, loads its
# data, runs the chained pooled-bootstrap + chained circular shuffle +
# label-permutation test (same flow as cells 2-3), and aggregates the
# per-animal pop_corr_A / pop_corr_B observed means plus the two-tailed
# permutation p for the A-vs-B difference.
#
# Slope plot: one line per animal connecting `pop_corr(group_A)` (left)
# to `pop_corr(group_B)` (right). Significant animals (p_two<0.05) are
# rendered in `GROUP_A_COLOR` if A > B, `GROUP_B_COLOR` if B > A; n.s.
# animals are gray. The aggregate mean line is overlaid in
# `THRESHOLD_COLOR`.
#
# Hyperparameters are reused from cell 2.

cross_animal_results: dict[str, dict] = {}
for animal_id, session_names in ANIMALS_TO_SESSIONS.items():
    print(f"Animal {animal_id} ({len(session_names)} sessions) ...", flush=True)
    animal_sessions = load_animal_sessions(animal_id, session_names)
    if not animal_sessions:
        print(f"  no sessions loaded, skipping")
        continue

    # Pool count matrices across the animal's sessions.
    a_mats, b_mats = [], []
    for sess in animal_sessions:
        a_starts = sess['group_a_df']['start'].to_numpy()
        b_starts = sess['group_b_df']['start'].to_numpy()
        if len(a_starts) < 1 or len(b_starts) < 1:
            continue
        a_mats.append(engine.extract_snippet_matrix(a_starts, sess['neural_data'], WINDOW_S))
        b_mats.append(engine.extract_snippet_matrix(b_starts, sess['neural_data'], WINDOW_S))
    if not a_mats or not b_mats:
        print(f"  insufficient trials, skipping")
        continue
    pooled_a = np.hstack(a_mats)
    pooled_b = np.hstack(b_mats)

    # Bootstrap-matched pop_corr means.
    bootstrap_target = min(USV_BOOTSTRAP_NUM, pooled_a.shape[1], pooled_b.shape[1])
    boot_a = engine.bootstrap_coactivity_distribution(pooled_a, bootstrap_target, N_BOOT_ITERATIONS)
    boot_b = engine.bootstrap_coactivity_distribution(pooled_b, bootstrap_target, N_BOOT_ITERATIONS)
    pop_a_obs = float(np.mean(boot_a['pop_corr']))
    pop_b_obs = float(np.mean(boot_b['pop_corr']))

    # Label permutation test on the pooled trials (direct A vs B test).
    perm = engine.perform_label_permutation_test(
        counts_a=pooled_a, counts_b=pooled_b, n_permutations=N_PERMUTATIONS,
    )
    p_two = perm['pop_corr']['p_two_tailed']
    p_a_gt_b = perm['pop_corr']['p_a_gt_b']
    z_score = perm['pop_corr']['z_score']

    cross_animal_results[animal_id] = {
        'n_sessions':   len(animal_sessions),
        'n_a':          pooled_a.shape[1],
        'n_b':          pooled_b.shape[1],
        'n_units':      pooled_a.shape[0],
        'pop_a':        pop_a_obs,
        'pop_b':        pop_b_obs,
        'p_two':        p_two,
        'p_a_gt_b':     p_a_gt_b,
        'z':            z_score,
    }
    print(
        f"  units={pooled_a.shape[0]:>3}  n_a={pooled_a.shape[1]:>5}  n_b={pooled_b.shape[1]:>5}"
        f"  pop_a={pop_a_obs:+.4f}  pop_b={pop_b_obs:+.4f}"
        f"  Δ={pop_a_obs-pop_b_obs:+.4f}  p_two={p_two:.3f}  Z={z_score:+.2f}"
    )

# Slope plot
fig, ax = plt.subplots(figsize=(6.5, 5.0))
sig_alpha = 0.05
for animal_id, r in cross_animal_results.items():
    delta = r['pop_a'] - r['pop_b']
    if r['p_two'] < sig_alpha:
        line_color = GROUP_A_COLOR if delta > 0 else GROUP_B_COLOR
        line_alpha = 0.95
    else:
        line_color = NULL_COLOR
        line_alpha = 0.55
    ax.plot([0, 1], [r['pop_a'], r['pop_b']], color=line_color, alpha=line_alpha, linewidth=2.0, marker='o')
    ax.text(
        1.03, r['pop_b'],
        f"{animal_id}  (p={r['p_two']:.3f}{', *' if r['p_two'] < sig_alpha else ''})",
        fontsize=8, va='center', color=line_color,
    )

# Aggregate mean line
mean_a = float(np.mean([r['pop_a'] for r in cross_animal_results.values()]))
mean_b = float(np.mean([r['pop_b'] for r in cross_animal_results.values()]))
ax.plot([0, 1], [mean_a, mean_b], color=THRESHOLD_COLOR, linewidth=3.0, linestyle='--', label=f"mean of animals")
ax.set_xticks([0, 1])
ax.set_xticklabels([f"{GROUP_A_LABEL}\n(IDs={GROUP_A_IDS})", f"{GROUP_B_LABEL}\n(IDs={GROUP_B_IDS})"])
ax.set_xlim(-0.15, 1.6)
ax.set_ylabel('pop_corr (bootstrap mean)', fontsize=10)
ax.set_title(
    f"Cross-animal pop_corr  ·  {CATEGORY_COLUMN}  ·  "
    f"N={len(cross_animal_results)} mice",
    fontsize=11,
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()